# 用 train.jsonl 训练 Qwen2.5-1.5B 并导出 GGUF

本 notebook 会训练轻量 LoRA、合并模型、转换为 GGUF Q4_K_M，并上传到 Hugging Face。只在输入框中粘贴 Token，不要把 Token 写进代码。

In [ ]:
!pip -q install -U unsloth datasets transformers==4.51.3 trl==0.15.2 peft accelerate bitsandbytes huggingface_hub
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!pip -q install -r /content/llama.cpp/requirements.txt

In [ ]:
from google.colab import files
uploaded = files.upload()
assert 'train.jsonl' in uploaded, '请上传 train.jsonl'

from getpass import getpass
HF_TOKEN = getpass('粘贴 Hugging Face 写入 Token（不会显示）：')

In [ ]:
from datasets import load_dataset
from unsloth import FastLanguageModel

dataset = load_dataset('json', data_files='train.jsonl', split='train')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit',
    max_seq_length=2048,
    load_in_4bit=True,
    token=HF_TOKEN,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
)

In [ ]:
def to_text(example):
    return {'text': tokenizer.apply_chat_template(example['messages'], tokenize=False, add_generation_prompt=False)}
dataset = dataset.map(to_text)

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    dataset_text_field='text', max_seq_length=2048,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=1, learning_rate=2e-4, warmup_ratio=0.05,
        logging_steps=10, save_strategy='no', report_to='none',
        fp16=True, optim='adamw_8bit', output_dir='/content/out',
    ),
)
trainer.train()

In [ ]:
merged_dir = '/content/qwen15b-style-merged'
model.save_pretrained_merged(merged_dir, tokenizer, save_method='merged_16bit')
!python /content/llama.cpp/convert_hf_to_gguf.py {merged_dir} --outfile /content/qwen15b-style-f16.gguf --outtype f16
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_NATIVE=OFF -DCMAKE_BUILD_TYPE=Release >/dev/null
!cmake --build /content/llama.cpp/build --config Release -j 2 --target llama-quantize >/dev/null
!mkdir -p /content/gguf
! /content/llama.cpp/build/bin/llama-quantize /content/qwen15b-style-f16.gguf /content/gguf/qwen15b-style-Q4_K_M.gguf Q4_K_M

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
repo_id = 'fengminqi/my-tg-qwen1.5b-style'
api.create_repo(repo_id, repo_type='model', exist_ok=True, private=True)
api.upload_file(
    path_or_fileobj='/content/gguf/qwen15b-style-Q4_K_M.gguf',
    path_in_repo='qwen15b-style-Q4_K_M.gguf',
    repo_id=repo_id, repo_type='model', token=HF_TOKEN,
)
print('上传完成:', f'https://huggingface.co/{repo_id}')